# Per-Run Budget Cap with LiteLLM + veronica-core

LiteLLM reports `response_cost` after each successful call.
This notebook uses a custom callback to record that cost in a
[veronica-core](https://github.com/amabito/veronica-core) `BudgetEnforcer`,
so the outer loop can stop before the next call once the run budget is exhausted.

In [ ]:
!pip install -q litellm veronica-core

## What this shows

- A `CustomLogger` callback records each call's cost via `BudgetEnforcer.spend()`.
- The loop checks `budget.is_exceeded` **before** the next call and stops.
- No changes to LiteLLM internals -- callback only.

In [ ]:
import os
import litellm
from litellm.integrations.custom_logger import CustomLogger
from veronica_core import BudgetEnforcer

# Set your API key (or use any provider LiteLLM supports).
# os.environ["OPENAI_API_KEY"] = "sk-..."

budget = BudgetEnforcer(limit_usd=0.05)


class RunBudgetCallback(CustomLogger):
    """Record response_cost in BudgetEnforcer after each successful call."""

    def log_success_event(self, kwargs, response_obj, start_time, end_time):
        cost = kwargs.get("response_cost", 0.0)
        if cost > 0:
            budget.spend(cost)


litellm.callbacks = [RunBudgetCallback()]
print(f"Run budget: ${budget.limit_usd:.2f}")

In [ ]:
MAX_CALLS = 20

for i in range(MAX_CALLS):
    if budget.is_exceeded:
        print(f"\nStopped before call {i}: run budget exhausted.")
        break

    response = litellm.completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Say 'hello' in one word. ({i})"}],
        max_tokens=10,
    )
    print(f"call {i:>2d}  spent=${budget.spent_usd:.4f}  "
          f"remaining=${budget.remaining_usd:.4f}  "
          f"calls={budget.call_count}")

print(f"\nFinal: ${budget.spent_usd:.4f} / ${budget.limit_usd:.2f}  "
      f"({budget.call_count} calls)")

## Next steps

veronica-core also provides `CircuitBreaker` for failure isolation
and `ExecutionContext` for combined cost / step / retry / timeout limits.
See the [README](https://github.com/amabito/veronica-core) for details.

> **Note:** `BudgetEnforcer.spend()` is post-call accounting.
> The call that triggers the limit has already completed;
> the loop stops before the *next* call.